In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

In [15]:
movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')
print(movies.head())
print(ratings.head())

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931


In [6]:
data = pd.merge(ratings, movies, on='movieId')
print(data)

        userId  movieId  rating   timestamp                           title  \
0            1        1     4.0   964982703                Toy Story (1995)   
1            1        3     4.0   964981247         Grumpier Old Men (1995)   
2            1        6     4.0   964982224                     Heat (1995)   
3            1       47     5.0   964983815     Seven (a.k.a. Se7en) (1995)   
4            1       50     5.0   964982931      Usual Suspects, The (1995)   
...        ...      ...     ...         ...                             ...   
100831     610   166534     4.0  1493848402                    Split (2017)   
100832     610   168248     5.0  1493850091   John Wick: Chapter Two (2017)   
100833     610   168250     5.0  1494273047                  Get Out (2017)   
100834     610   168252     5.0  1493846352                    Logan (2017)   
100835     610   170875     3.0  1493846415  The Fate of the Furious (2017)   

                                             genres

In [7]:
pivot = data.pivot_table(index='title', columns='userId', values='rating').fillna(0)

print(pivot.head())

userId                                   1    2    3    4    5    6    7    \
title                                                                        
'71 (2014)                               0.0  0.0  0.0  0.0  0.0  0.0  0.0   
'Hellboy': The Seeds of Creation (2004)  0.0  0.0  0.0  0.0  0.0  0.0  0.0   
'Round Midnight (1986)                   0.0  0.0  0.0  0.0  0.0  0.0  0.0   
'Salem's Lot (2004)                      0.0  0.0  0.0  0.0  0.0  0.0  0.0   
'Til There Was You (1997)                0.0  0.0  0.0  0.0  0.0  0.0  0.0   

userId                                   8    9    10   ...  601  602  603  \
title                                                   ...                  
'71 (2014)                               0.0  0.0  0.0  ...  0.0  0.0  0.0   
'Hellboy': The Seeds of Creation (2004)  0.0  0.0  0.0  ...  0.0  0.0  0.0   
'Round Midnight (1986)                   0.0  0.0  0.0  ...  0.0  0.0  0.0   
'Salem's Lot (2004)                      0.0  0.0  0.0  ...  0.

In [8]:
model = NearestNeighbors(metric='cosine', algorithm='brute')
model.fit(pivot)

NearestNeighbors(algorithm='brute', metric='cosine')

In [9]:
def rekomendasi_film(judul_film, jumlah_rekomendasi=5):
    if judul_film not in pivot.index:
        return "Film tidak ditemukan"

    index = pivot.index.get_loc(judul_film)
    jarak, indeks = model.kneighbors([pivot.iloc[index]], n_neighbors=jumlah_rekomendasi+1)

    hasil = []
    for i in range(1, len(indeks[0])):
        hasil.append(pivot.index[indeks[0][i]])

    return hasil

In [10]:
film = pivot.index[0]  # ambil salah satu film
print("Film input:", film)

rekomendasi = rekomendasi_film(film)
print("Rekomendasi film:")
for i, f in enumerate(rekomendasi):
    print(f"{i+1}. {f}")

Film input: '71 (2014)
Rekomendasi film:
1. City of Lost Souls, The (Hyôryuu-gai) (2000)
2. Reality (2014)
3. Cooties (2015)
4. Rapture-Palooza (2013)
5. Red Hill (2010)


# Analisis Model K-Nearest Neighbor (KNN)

Pada percobaan ini, metode K-Nearest Neighbor (KNN) digunakan untuk membangun sistem rekomendasi sederhana berdasarkan data film dan rating pengguna. Berbeda dengan metode regresi yang berfokus pada prediksi nilai, KNN bekerja dengan mencari data yang memiliki kemiripan paling dekat dengan data yang dipilih. Dalam kasus ini, setiap film dibandingkan berdasarkan pola rating dari banyak pengguna. Dengan pendekatan tersebut, sistem dapat memberikan rekomendasi film lain yang kemungkinan memiliki karakteristik atau minat penonton yang serupa.

Proses awal dimulai dengan menggabungkan dua file CSV yaitu `movies.csv` dan `ratings.csv`. File pertama berisi informasi daftar film, sedangkan file kedua berisi rating yang diberikan pengguna terhadap film tertentu. Setelah data digabungkan, dibuat sebuah pivot table yang mengubah data menjadi bentuk matriks antara judul film dan pengguna. Nilai rating yang kosong diisi dengan angka nol agar model dapat menghitung jarak antar data secara konsisten. Tahap ini penting karena KNN membutuhkan data numerik yang tersusun rapi sebelum proses pencarian tetangga terdekat dilakukan.

Pada implementasinya, model menggunakan cosine similarity untuk mengukur tingkat kedekatan antar film. Semakin kecil jaraknya, maka semakin mirip pola rating film tersebut. Saat satu film dipilih sebagai input, model akan mencari beberapa film lain yang memiliki pola penilaian paling mirip dari pengguna sebelumnya. Hasil ini dapat digunakan sebagai rekomendasi otomatis. Misalnya, jika pengguna menyukai satu film tertentu, maka sistem dapat menampilkan beberapa film lain yang kemungkinan juga disukai oleh pengguna tersebut berdasarkan perilaku rating pengguna lain.

Dari hasil analisis, KNN cocok digunakan untuk sistem rekomendasi karena sederhana namun cukup efektif dalam mengenali pola kemiripan data. Kelebihan utamanya adalah model ini tidak membutuhkan proses pelatihan yang rumit seperti algoritma lain. Namun, kekurangannya adalah performa dapat melambat jika jumlah data semakin besar karena sistem harus menghitung jarak ke banyak data sekaligus. Dalam konteks Sistem Informasi atau bisnis digital, metode ini sangat relevan untuk diterapkan pada fitur rekomendasi produk, film, atau konten yang ingin ditampilkan secara personal kepada pengguna.